In [1]:
from nupack import *
import subprocess
import tempfile
import os, sys
import random
random.seed(102)
import pickle
import copy
import time
from datetime import datetime
import numpy as np
import pandas as pd
import scadnano as sc # need scadnano python package installed

In [2]:
# This suppresses the annoying WARNING statements for duplicate slat names (from stackoverflow)
class NoMoreWarnies:
    def __enter__(self):
        self._original_stdout = sys.stdout
        sys.stdout = open(os.devnull, 'w')

    def __exit__(self, exc_type, exc_val, exc_tb):
        sys.stdout.close()
        sys.stdout = self._original_stdout

In [3]:
scadnano_fp = '240524 v0b seqs.sc' # filepath to scadnano design
label = 'v0b' # label to identify design by in generated files
idt_label = 'v0b'
it_num = 5000 # number of iterations of sequence optimisation algorithm
num_x = 6 # number of unique x-slats
num_y = 6 # number of unique y-slats
num_t = 7 # change to desired T-brush length

In [4]:

x_0 = [(5,6,5,6,5,6,5,6,5,6,5,6)]
x_1 = [(5,6,5,6,5,6,5,6,5,6,5,6)]
x_2 = [(5,6,5,6,5,6,5,6,5,6,5,6)]
x_3 = [(5,6,5,6,5,6,5,6,5,6,5,6)]
x_4 = [(5,6,5,6,5,6,5,6,5,6,5,6)]
x_5 = [(5,6,5,6,5,6,5,6,5,6,5,6)]

In [5]:
# slat names from design file
x_name = 'x'
y_name = 'y'


If you are running nupack 4 please add above: from nupack import *
& change MFE calculating function to:

    def self_dG_FN(seq):
    
        '''Calculates MFE of single DNA sequence using NUPACK'''    
    
        # Parameters for free energy calculations
        # Lowest sodium value is 0.05 in nupack 4
        modely = Model(material='dna', celsius=45 ,sodium = 0.05, magnesium=0.015)
        #
        a = Strand(seq, name='a')
        c = Complex([a], name='c')
        t1 = Tube(strands={a: 1}, complexes=SetSpec(max_size=1, include=[c]), name='t1')
        results = tube_analysis(tubes=[t1], model=modely,compute=['mfe'])
        energy = results[c].mfe[0].energy
    
        if energy > 0:
            energy = 0
        return energy

This will change the energy values that are calculated when compared to the values generated below. I compared the generated values with the nupack web-based online calculation and they seem roughly similar.

In [6]:
# Parameters for free energy calculations
def self_dG_FN(seq):

    '''Calculates MFE of single DNA sequence using NUPACK'''    

    # Parameters for free energy calculations
    # Lowest sodium value is 0.05 in nupack 4
    modely = Model(material='dna', celsius=45 ,sodium = 0.05, magnesium=0.015)
    #
    a = Strand(seq, name='a')
    c = Complex([a], name='c')
    t1 = Tube(strands={a: 1}, complexes=SetSpec(max_size=1, include=[c]), name='t1')
    results = tube_analysis(tubes=[t1], model=modely,compute=['mfe'])
    energy = results[c].mfe[0].energy

    if energy > 0:
        energy = 0
    return energy

In [7]:
def rcomp(seq):
    
    '''Generates reverse complement of DNA sequence'''
    
    complement = {'A': 'T', 'T': 'A', 'C': 'G', 'G': 'C'} 
    return ''.join([complement[base] for base in reversed(seq.upper())])

In [8]:
with open('designed_seqs_05-69.pkl', 'rb') as words_file: # library of isoenergetic sequences of 5-69 nt in length
    des_words = pickle.load(words_file)

In [9]:
def gen_y(x_words, x_name, y_name):
    
    x_seqs = [''.join(s) for s in x_words]
    
    with NoMoreWarnies(): # removes annoying WARNING statement for duplicate names
        cc = sc.Design()
        cc = cc.from_scadnano_file(scadnano_fp)
        
    # assign x-sequences to scadnano design    
    for i in range(num_x):
        name = x_name + str(i)
        for strand in cc.strands:
            if strand.name == name:
                cc.assign_dna(strand=strand,sequence=x_seqs[i])
    
    # get y-sequences from scadnano design
    y_seqs = []
    for i in range(num_y):
        name = y_name + str(i)
        for strand in cc.strands:
            if strand.name == name:
                y_seqs.append(strand.dna_sequence)
    
    return y_seqs

In [10]:
def gen_nuc_y(x_seqs, x_name, n_name):
    
    with NoMoreWarnies(): # removes annoying WARNING statement for duplicate names
        cc = sc.Design()
        cc = cc.from_scadnano_file(scadnano_fp)
    
    # assign x-sequences to scadnano design    
    for i in range(num_x):
        name = x_name + str(i)
        for strand in cc.strands:
            if strand.name == name:
                cc.assign_dna(strand=strand,sequence=x_seqs[i])
    
    # get nuc-y sequences from scadnano design
    nuc_y_seqs = []
    for i in range(num_ny):
        name = n_name + str(i)
        for strand in cc.strands:
            if strand.name == name:
                nuc_y_seqs.append(strand.dna_sequence)
    
    return nuc_y_seqs

In [11]:
def score_self_structure(x_words,y_seqs):
    
    # Calculate self-structure free energies for all x and y slats and sum
    dG_x = [self_dG_FN(''.join(slat)) for slat in x_words]
    dG_y = [self_dG_FN(slat) for slat in y_seqs]
    dG = np.sum(dG_x) + np.sum(dG_y)
    
    return dG

In [12]:
# From Protozanova paper
dinuc_dG = {'AA': -1.11, 'AC': -1.81, 'AG': -1.06, 'AT': -1.34, 'CA': -0.55, 'CC': -1.44, 'CG': -0.91, 'CT': -1.06,
            'GA': -1.43, 'GC': -2.17, 'GG': -1.44, 'GT': -1.81, 'TA': -0.19, 'TC': -1.43, 'TG': -0.55, 'TT': -1.11}

def score_x_stacking(x_words): # Super preliminary, please ignore for now
    
    x_stack = [[dinuc_dG[slat[j][-1] + slat[j+1][0]] if i%2 == 0 else dinuc_dG[slat[-(j+1)][0] + slat[-(j+2)][-1]] for j in range(len(slat)-1)] for i,slat in enumerate(x_words)]
    
    return np.sum(x_stack)

In [13]:
def score_seqs(x_words,y_seqs): # Scoring function for comparing sequences
    
    return score_self_structure(x_words,y_seqs) # Currently only considering self structure

In [14]:
# run algorithm N times for N variants
all_sequences = []
versions = []
for k in range(3):
    
    start = time.time()

    max_score = -np.inf

    # Generate new random set of slats and update sequences if that reduces self-structure free energy
    for i in range(it_num):
        if i%20 == 0:
            print(f'Current iteration: {i} \t time (s): {time.time()-start}')
        x_words = []
        # Generate new random sequences
        for index in range(0,6):
            list_name_x = f'x_{index}'
            list_name_xwords = f'x_words{index}'
            globals()[list_name_xwords] = [[random.choice(des_words[w]) if w == 5 or w == 6 else w for i,w in enumerate(slat)] for slat in globals()[list_name_x]]
            x_words += globals()[list_name_xwords]
    
        y_seqs = gen_y(x_words, x_name, y_name)
        score = score_seqs(x_words, y_seqs)
    
        if score > max_score:
            print(f'Swap iteration: {i} \t new score: {score}')
            max_score = score
            max_x_words = x_words
            max_y_seqs = y_seqs
        

    print(f'\nTotal time taken (s): {time.time()-start} \t end score: {max_score}')

    
  
    # join x sequences
    x_seqs_words_check = max_x_words
    x_seqs = [''.join(s) for s in max_x_words]
    
    # sequences
    x = [x_seqs[s] for s in range(len(x_seqs))]
    y = [s for i,s in enumerate(max_y_seqs)]

    
 
    all_sequences.append(x)
    all_sequences.append(y)
    
    # make label for each version
    version = 'v'+str(k)
    labels = label+'_v'+str(k)
    
    # versions
    versions.append(version)

    # Create scadnano file with sequences from final design – need to manually change 'grid' to 'square' after saving
    with NoMoreWarnies(): # removes annoying WARNING statement for duplicate names
        cc = sc.Design()
        cc = cc.from_scadnano_file(scadnano_fp)

    # assign x-sequences to scadnano design    
    for i in range(num_x):
        name = x_name + str(i)
        for strand in cc.strands:
            if strand.name == name:
                cc.assign_dna(strand=strand,sequence=x_seqs[i])

    with NoMoreWarnies(): # removes annoying WARNING statement for duplicate names            
        cc.set_grid(sc.Grid('square'))
        cc.write_scadnano_file(filename=datetime.now().strftime('%Y%m%d_{}.dna').format(labels))


Current iteration: 0 	 time (s): 1.0967254638671875e-05
Swap iteration: 0 	 new score: -33.81194305419922
Swap iteration: 1 	 new score: -32.79795813560486
Swap iteration: 5 	 new score: -28.190601110458374
Current iteration: 20 	 time (s): 13.5493483543396
Swap iteration: 22 	 new score: -25.721261501312256
Current iteration: 40 	 time (s): 26.731221437454224
Current iteration: 60 	 time (s): 39.893500328063965
Current iteration: 80 	 time (s): 52.397093057632446
Current iteration: 100 	 time (s): 64.57799172401428
Current iteration: 120 	 time (s): 76.55725073814392
Current iteration: 140 	 time (s): 88.66307401657104
Current iteration: 160 	 time (s): 100.615553855896
Current iteration: 180 	 time (s): 112.92997312545776
Current iteration: 200 	 time (s): 125.12951016426086
Current iteration: 220 	 time (s): 137.34046053886414
Current iteration: 240 	 time (s): 150.12359142303467
Current iteration: 260 	 time (s): 162.69641757011414
Current iteration: 280 	 time (s): 175.11322617530

In [15]:
x_seqs = [''.join(s) for s in max_x_words]
x = [num_t*'t'+s+num_t*'t' for i, s in enumerate(x_seqs)] # add WEST T-brush to x-slats
y = [s for i,s in enumerate(max_y_seqs)] # add SOUTH T-brush to y-slats

In [16]:
# Create scadnano file with sequences from final design – need to manually change 'grid' to 'square' after saving

with NoMoreWarnies(): # removes annoying WARNING statement for duplicate names
    cc = sc.Design()
    cc = cc.from_scadnano_file(scadnano_fp)
        
# assign x-sequences to scadnano design    
for i in range(num_x):
    name = x_name + str(i)
    for strand in cc.strands:
        if strand.name == name:
            cc.assign_dna(strand=strand,sequence=x_seqs[i])

with NoMoreWarnies(): # removes annoying WARNING statement for duplicate names            
    cc.set_grid(sc.Grid('square'))
    cc.write_scadnano_file(filename=datetime.now().strftime('%Y%m%d_{}.dna').format(label))

In [17]:
x

['tttttttGTGACTAATGCCCATCGCCTAAAAGAAGACTGAACTCCGGAACGCAGCTATGCAAGCTACTAAGGGttttttt',
 'tttttttAGAAGTAAGCTTTCTGTTTCCGCATGGTGGCTATAAGCCGCAAAAGTTCCCGAATATGCAGGTGTAttttttt',
 'tttttttAGCAGACAGGACAGAGAGGTGTAGCATATCCGTACCATCGCAAAATTGAGCCCTACTTCGCATGCTttttttt',
 'tttttttCTCACTGACCATCTGTATTCCTATGAGGCATAGTGTGTGTTTTGACAGGTGAGTCGCAGTAGATGTttttttt',
 'tttttttTAGTCGCTTTGCCTACGAGTCGACAGGTATTCGGGAAGCAAGTGATCACCCCAAACCTCTCTTCGTttttttt',
 'tttttttTCACATGTTCATCGTTCAGACTTTCAGAGTGAATGTCCAGGTGTTCCTCGATCCCGCTTCAGGTCAttttttt']

In [18]:
y

['AGCATGCTTCTCCCTTAGGACACGAATAACACAACGGATGCTTATCAGTCTGTGAACGAAGGTGAG',
 'AGCATGCGAAGTGGTCAAGAGGTGAACATTCTTTTTGCGATGCTCAAAACCCTGTACACCTGTAGCAGCTTACGAAG',
 'AGGAATACAGATAGGGCCAGAATTGCATGAGGACGACTCCCTGTACACCTGAACTTTAGGCAACGATTTGGGACAGA',
 'AGGAATGTGATAGTCTGGATGGATTCGGCTCTGGACTCAGTAGGGGGATCAGCTGCGGAAATCAAT',
 'TTTGCGCCATGCGTTCCGAAGCCAAAGCACTGCTCCTGTTGCATGCATTACTGAACACTTGCTCATTTTGCGATGGT',
 'CTATGCCTTCCTTCACTGTCACTACACCCTGCTACATCTGACTATGACCTGGAGTTAGCCAATGGT']